# Step 6 — Feature Engineering
**Project**: Deep Learning-Based Flood Prediction Using Rainfall Data  
**Dataset**: `data/raw/flood_risk_dataset_india.csv` (10,000 spatial observations across India)  
**Objective**: Construct domain-specific hydrological/meteorological interaction features, physical categorical encodings, vulnerability indices, leakage audit, stratified train/test split, and model input preparation for ML & LSTM.

In [1]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Helper for display in IPython / fallback
try:
    from IPython.display import display
except ImportError:
    def display(df):
        print(df)

# Add project root directory to path
sys.path.append('..')
from src.feature_engineering import (
    normalize_column_names,
    review_existing_features,
    create_all_engineered_features,
    select_and_filter_features,
    prepare_model_inputs,
    reshape_for_lstm,
    save_engineered_data
)

# Set visualization style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

## 1. Review Existing Features
Audit raw dataset schema, identify feature modalities (numerical, categorical, spatial coordinates, target variable), and verify presence/absence of temporal attributes.

In [2]:
df_raw = pd.read_csv('../data/raw/flood_risk_dataset_india.csv')
review = review_existing_features(df_raw)
print('--- Dataset Audit Summary ---')
for key, val in review.items():
    print(f"{key:25s}: {val}")

df_norm = normalize_column_names(df_raw)
print('\nNormalized DataFrame Head:')
print(df_norm.head())

--- Dataset Audit Summary ---
total_columns            : 14
target_column            : Flood_Occurred
time_columns             : []
has_time_dimension       : False
numerical_features       : ['Latitude', 'Longitude', 'Rainfall_mm', 'Temperature_C', 'Humidity_pct', 'River_Discharge_m3s', 'Water_Level_m', 'Elevation_m', 'Population_Density', 'Infrastructure', 'Historical_Floods']
categorical_features     : ['Land_Cover', 'Soil_Type']
total_records            : 10000

Normalized DataFrame Head:
    Latitude  Longitude  ...  Historical_Floods  Flood_Occurred
0  18.861663  78.835584  ...                  0               1
1  35.570715  77.654451  ...                  1               0
2  29.227824  73.108463  ...                  1               1
3  25.361096  85.610733  ...                  1               0
4  12.524541  81.822101  ...                  0               0

[5 rows x 14 columns]


## 2. Numerical Feature Engineering
Engineering physical interaction features derived from hydro-meteorological relationships:
- **Rainfall-to-Elevation Ratio**: Measures heavy rainfall accumulation potential relative to terrain elevation.
- **Hydro Load Index**: Combined pressure of atmospheric precipitation (`Rainfall_mm`) and surface streamflow (`River_Discharge_m3s`) normalized by elevation.
- **Discharge-per-Water-Level Ratio**: Measures channel flow velocity/capacity efficiency.
- **Relative Humidity Ratio**: Moisture content normalized by ambient surface temperature.
- **Elevation Inverse**: Quantifies lowland inundation susceptibility.

In [3]:
df_eng = create_all_engineered_features(df_raw)
print(f"Raw Features Count      : {df_norm.shape[1] - 1}")
print(f"Total Engineered Columns: {df_eng.shape[1]}")

new_hydro_cols = ['Rainfall_Elevation_Ratio', 'Hydro_Load_Index', 'Discharge_WaterLevel_Ratio', 'Relative_Humidity_Ratio', 'Elevation_Inverse']
print('\nSummary of Engineered Hydrological Features:')
print(df_eng[new_hydro_cols].describe().T[['mean', 'std', 'min', '50%', 'max']])

Raw Features Count      : 13
Total Engineered Columns: 31

Summary of Engineered Hydrological Features:
                                   mean           std  ...         50%           max
Rainfall_Elevation_Ratio       0.152036      1.468868  ...    0.033840  8.501747e+01
Hydro_Load_Index             289.408770   1590.404758  ...   72.092217  7.162297e+04
Discharge_WaterLevel_Ratio  1908.262493  15149.501159  ...  504.093242  1.061158e+06
Relative_Humidity_Ratio        2.103521      1.066489  ...    1.938706  6.104962e+00
Elevation_Inverse              0.001057      0.009953  ...    0.000226  4.650428e-01

[5 rows x 5 columns]


## 3. Categorical Encodings & Socioeconomic Vulnerability Features
- **Soil Permeability Index**: Ordinal physical mapping based on soil drainage (Clay=3.0 [impermeable], Peat=2.5, Silty=2.0, Loam=1.5, Alluvial=1.5, Sandy=1.0 [permeable]).
- **Land Cover Inundation Risk**: Ordinal physical score based on runoff coefficients (Water Body=3.0, Urban=2.5, Agriculture=2.0, Grassland=1.5, Forest=1.0).
- **One-Hot Encoding**: Unbiased binary indicator variables for `Land_Cover` and `Soil_Type`.
- **Vulnerability Interaction Indices**: `Infrastructure_Vulnerability` (`Population_Density` x `Infrastructure`) and `Historical_Rainfall_Interaction` (`Historical_Floods` x `Rainfall_mm`).

In [4]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=df_eng, x='Flood_Occurred', y='Soil_Permeability_Index', palette='Blues', ax=ax[0], hue='Flood_Occurred', legend=False)
ax[0].set_title('Soil Permeability Index vs. Flood Occurrence', fontsize=12, fontweight='bold')
ax[0].set_xticks([0, 1])
ax[0].set_xticklabels(['No Flood (0)', 'Flood (1)'])

sns.boxplot(data=df_eng, x='Flood_Occurred', y='Hydro_Load_Index', palette='Oranges', ax=ax[1], hue='Flood_Occurred', legend=False)
ax[1].set_title('Hydro Load Index vs. Flood Occurrence', fontsize=12, fontweight='bold')
ax[1].set_xticks([0, 1])
ax[1].set_xticklabels(['No Flood (0)', 'Flood (1)'])
plt.tight_layout()
plt.show()

## 4. Policy on Time-Based, Lag & Rolling Features
> **Strict Rule Compliance**: The dataset consists of 10,000 spatial observations across India without explicit timestamps/date records. Per project guidelines (*'Do not create rolling or lag features by randomly ordering rows'*), **artificial lag and rolling windows are NOT created** to prevent data leakage and false sequence assumptions.

## 5. Feature Selection & Data Leakage Audit
Examine correlation matrix across engineered features to detect multicollinearity ($r > 0.95$) or target leakage.

In [5]:
X_raw, y_raw, dropped_cols = select_and_filter_features(df_eng, target_col='Flood_Occurred', corr_threshold=0.95)
print(f"Target Column Excluded from Features : 'Flood_Occurred'")
print(f"Collinear Features Dropped (>0.95)   : {dropped_cols}")
print(f"Final Input Feature Count            : {X_raw.shape[1]}")

# Correlation with target
corr_with_target = df_eng.corr(numeric_only=True)['Flood_Occurred'].sort_values(ascending=False)
print("\nTop 10 Feature Correlations with Flood Target:")
print(corr_with_target.head(11))

Target Column Excluded from Features : 'Flood_Occurred'
Collinear Features Dropped (>0.95)   : []
Final Input Feature Count            : 30

Top 10 Feature Correlations with Flood Target:
Flood_Occurred                1.000000
Humidity_pct                  0.027754
Relative_Humidity_Ratio       0.026430
Discharge_WaterLevel_Ratio    0.021840
Hydro_Load_Index              0.013033
Historical_Floods             0.012030
Soil_Type_Clay                0.011913
Land_Cover_Desert             0.010393
Elevation_Inverse             0.010371
Land_Cover_Urban              0.009591
Land_Cover_Agricultural       0.008398
Name: Flood_Occurred, dtype: float64


## 6. Train / Test Split & Scaler Leakage Prevention
Apply **Stratified Train/Test Split (80/20)** and fit `StandardScaler` **only on $X_{train}$**, then transform $X_{train}$ and $X_{test}$.

In [6]:
X_train, X_test, y_train, y_test, feature_names, scaler = prepare_model_inputs(df_eng, target_col='Flood_Occurred', test_size=0.2, random_state=42)
print(f"X_train Shape : {X_train.shape}")
print(f"X_test Shape  : {X_test.shape}")
print(f"y_train Shape : {y_train.shape}")
print(f"y_test Shape  : {y_test.shape}")

# Verify Scaling
print(f"X_train Mean (sample) : {X_train.mean().values[0]:.4f} (expected ~0.0)")
print(f"X_train Std  (sample) : {X_train.std().values[0]:.4f} (expected ~1.0)")

X_train Shape : (8000, 30)
X_test Shape  : (2000, 30)
y_train Shape : (8000,)
y_test Shape  : (2000,)
X_train Mean (sample) : -0.0000 (expected ~0.0)
X_train Std  (sample) : 1.0001 (expected ~1.0)


## 7. Deep Learning / LSTM Input Reshaping
Reshape tabular matrices into 3D tensors `(N, timesteps=1, num_features)` required for LSTM / Deep Learning layers.

In [7]:
X_train_3d, X_test_3d = reshape_for_lstm(X_train, X_test, timesteps=1)
print(f"LSTM X_train 3D Shape: {X_train_3d.shape}")
print(f"LSTM X_test 3D Shape : {X_test_3d.shape}")

LSTM X_train 3D Shape: (8000, 1, 30)
LSTM X_test 3D Shape : (2000, 1, 30)


## 8. Save Processed Data & Final Feature Engineering Report

In [8]:
save_engineered_data(X_train, X_test, y_train, y_test, df_eng, output_dir='../data/processed')

print('='*60)
print('STEP 6 — FEATURE ENGINEERING & VALIDATION SUMMARY')
print('='*60)
print(f"1. Original Features Count  : {len(review['numerical_features']) + len(review['categorical_features'])}")
print(f"2. Engineered Features Count: {X_train.shape[1]}")
print(f"3. Target Column           : {y_train.name}")
print(f"4. Total Records            : {len(df_eng)}")
print(f"5. Training Set Samples     : {X_train.shape[0]} (80% Stratified Split)")
print(f"6. Testing Set Samples      : {X_test.shape[0]} (20% Stratified Split)")
print(f"7. Rows Removed (Lag/Roll)  : 0 (No artificial rows dropped; spatial data)")
print(f"8. LSTM Suitability         : Tabular Spatial data; 3D reshaped tensor (N, 1, 30) prepared")
print(f"9. Data Quality Status      : 0 missing values, 0 infinite values, 0 target leakage")
print('='*60)

STEP 6 — FEATURE ENGINEERING & VALIDATION SUMMARY
1. Original Features Count  : 13
2. Engineered Features Count: 30
3. Target Column           : Flood_Occurred
4. Total Records            : 10000
5. Training Set Samples     : 8000 (80% Stratified Split)
6. Testing Set Samples      : 2000 (20% Stratified Split)
7. Rows Removed (Lag/Roll)  : 0 (No artificial rows dropped; spatial data)
8. LSTM Suitability         : Tabular Spatial data; 3D reshaped tensor (N, 1, 30) prepared
9. Data Quality Status      : 0 missing values, 0 infinite values, 0 target leakage
